<a href="https://colab.research.google.com/github/joelnin-star/PI1-Bike-Sharing-Diseno-Experimental/blob/main/Checkpoint_Diseno_Experimental_Bike_Sharing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PROYECTO INTEGRADOR 1 – CHECKPOINT: DISEÑO EXPERIMENTAL
## Caso 3: Bike Sharing – Cuando la demanda cambia
### «Bicicletas donde y cuando se necesitan»: disponibilidad operativa frente a clima, calendario y hora

**Curso:** Analítica Empresarial Integrada  
**Carrera:** Big Data y Ciencia de Datos  
**Institución:** TECSUP  
**Periodo:** 2026-II  
**Ciclo:** 6  
**Docente:** Pilar Rocío Sayán Mejía  

### Integrantes
- Joel Ronald Niño de Guzmán Rodríguez
- Diego Yeferson Quispe Huamani
- Renzo Palomino


## 1. Cargar archivos


In [1]:
from google.colab import files
archivos = files.upload()


Saving day.csv to day.csv
Saving hour.csv to hour.csv


## 2. Leer los datos

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

hour = pd.read_csv('hour.csv')
day = pd.read_csv('day.csv')

print('hour.csv:', hour.shape)
print('day.csv:', day.shape)
display(hour.head())


hour.csv: (17379, 17)
day.csv: (731, 16)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## 3. Verificación rápida

In [4]:
# Comprobamos que el total sea igual a usuarios casuales + registrados
errores = (hour['cnt'] != hour['casual'] + hour['registered']).sum()
print('Filas donde cnt no coincide con casual + registered:', errores)


Filas donde cnt no coincide con casual + registered: 0


## 4. Línea base real

In [5]:
# Perfil promedio de demanda por hora del día
perfil_hora = hour.groupby('hr')['cnt'].mean()

# Consideramos horas de mayor demanda las que están en el 25 % superior del perfil horario
umbral_pico = perfil_hora.quantile(0.75)
horas_pico = perfil_hora[perfil_hora >= umbral_pico].index.tolist()

datos_pico = hour[hour['hr'].isin(horas_pico)].copy()
media_base = datos_pico['cnt'].mean()
desv_base = datos_pico['cnt'].std(ddof=1)

print('Horas de mayor demanda:', horas_pico)
print(f'Línea base promedio: {media_base:.2f} alquileres por hora')
print(f'Desviación estándar: {desv_base:.2f}')


Horas de mayor demanda: [8, 13, 16, 17, 18, 19]
Línea base promedio: 353.86 alquileres por hora
Desviación estándar: 208.14


## 6. Tamaño de muestra

In [9]:
import math

# Supuestos del diseño
alpha = 0.05       # 95 % de confianza
potencia = 0.80    # 80 % de potencia
mde_pct = 0.10     # detectar como mínimo una mejora del 10 %

delta = media_base * mde_pct
z_alpha = norm.ppf(1 - alpha / 2)
z_beta = norm.ppf(potencia)

# Aproximación para comparar dos medias independientes
n_por_grupo = math.ceil(2 * ((z_alpha + z_beta) ** 2) * (desv_base ** 2) / (delta ** 2))
n_total = n_por_grupo * 2

print(f'MDE usado: {delta:.2f} alquileres por hora ({mde_pct*100:.0f} % de la línea base)')
print(f'Muestra estimada por grupo: {n_por_grupo} observaciones')
print(f'Muestra total estimada: {n_total} observaciones')

MDE usado: 35.39 alquileres por hora (10 % de la línea base)
Muestra estimada por grupo: 544 observaciones
Muestra total estimada: 1088 observaciones


**Interpretación:** El cálculo utiliza una confianza del 95 %, potencia del 80 % y un cambio mínimo detectable del 10 %. La media y la variabilidad utilizadas provienen del dataset real. Este cálculo sirve como estimación inicial del piloto; el caso no contiene información por estación, por lo que en una aplicación real se necesitarían datos por estación o zona para definir las unidades experimentales finales.